In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
from bs4 import BeautifulSoup
import os
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pdfplumber
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pytesseract
from img2table.ocr import TesseractOCR
from img2table.document import Image
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR\tesseract.exe"
from img2table.ocr import TesseractOCR
from img2table.document import PDF
import cv2
import numpy as np
    

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'GN BCRG' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running GN BCRG Web Scraping Tool v.2.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,
         
		 "profile.default_content_setting_values.automatic_downloads": 1

         
		 }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   

   
regdict={
         regulatorName+' 1': 'https://www.bcrg-guinee.org/missions/supervision/banques/societes-bancaires/', 
         regulatorName+' 2': 'https://www.bcrg-guinee.org/missions/supervision/institutions-de-microfinances/listes-des-imf/', 
         regulatorName+' 3': 'https://www.bcrg-guinee.org/missions/supervision/assurances/liste-des-societes-dassurances/', 
        }




processdate = now.strftime('%Y-%m-%d')


print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))

The current folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\GN BCRG
The temp folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\GN BCRG\tempfolder


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def normalize_contact(text: str) -> str:
    # Remove bullet symbols and normalize line breaks / hyphen breaks
    text = text.replace("▪", " ")
    text = re.sub(r"-\s*\n\s*", "", text)  # merge hyphenated line breaks
    text = re.sub(r"\s*\n\s*", " ", text)  # newlines -> spaces
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

def extract_contacts(text: str):
    text = normalize_contact(text)

    # Emails
    emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)

    # Websites: capture www.* or http(s)://*
    websites = re.findall(r"(?:https?://|www\.)[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/[^\s,;]*)?", text)

    # Phones: capture Guinea patterns (with +224 / 00224 / 224 optional)
    # Allow separators: space, dash, slash, parentheses
    phone_candidates = re.findall(
        r"(?:\+|00)?\s*224[\s\-()./]*\d{2,3}[\s\-()./]*\d{2,3}[\s\-()./]*\d{2,3}|\b\d{2,3}[\s\-()./]*\d{2,3}[\s\-()./]*\d{2,3}\b",
        text
    )

    # Clean phone numbers: keep digits and leading + if present
    def clean_phone(p):
        p = p.strip()
        has_plus = "+" in p
        digits = re.sub(r"\D", "", p)
        if has_plus:
            return f"+{digits}"
        if digits.startswith("224"):
            return f"+{digits}"
        return digits

    phones = []
    for p in phone_candidates:
        cleaned = clean_phone(p)
        if cleaned not in phones:
            phones.append(cleaned)

    return {
        "phones": phones,
        "emails": list(dict.fromkeys(emails)),
        "websites": list(dict.fromkeys(websites)),
        "raw_cleaned": text,
    }



def normalize_address(text: str) -> str:
    # Remove bullet symbols and normalize line breaks / hyphen breaks
    text = text.replace("▪", " ")
    # Merge hyphenated line breaks: "San-\ndervalia" -> "Sandervalia"
    text = re.sub(r"-\s*\n\s*", "", text)
    # Replace remaining newlines with spaces
    text = re.sub(r"\s*\n\s*", " ", text)
    # Collapse multiple spaces
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

def extract_fields(text: str):
    text = normalize_address(text)

    # Commune: look for "Commune" or "commune"
    commune = ''
    m = re.search(r"\bCommune\s*[:\-]?\s*([A-Za-zÀ-ÿ' \-]+)", text, flags=re.IGNORECASE)
    if m:
        commune = m.group(1).strip()

    # BP / postal_code: capture numbers after BP or B.P.
    postal_code = ''
    m = re.search(r"\bB\.?\s*P\.?\s*[:\-]?\s*([0-9]+)", text, flags=re.IGNORECASE)
    if m:
        postal_code = m.group(1).strip()

    # City: usually "Conakry" appears; otherwise try after BP or at end
    city = ''
    m = re.search(r"\b(Conakry)\b", text, flags=re.IGNORECASE)
    if m:
        city = m.group(1)
    else:
        # Fallback: text after BP number
        m = re.search(r"\bB\.?\s*P\.?\s*[:\-]?\s*[0-9]+\s*[,/ -]*([A-Za-zÀ-ÿ' \-]+)", text, flags=re.IGNORECASE)
        if m:
            city = m.group(1).strip()

    return {
        "commune": commune,
        "city": city,
        "postal_code": postal_code,
        "raw_cleaned": text,
    }


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

In [6]:
# %%


#------------------------------------------------ Begin_ fileName ----------------------------------------

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(5)
    

    # the block where to find the links 
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Download the file
    wait = WebDriverWait(driver, 10)
    if  reg == 'GN BCRG 1':
        # Locate the download link (by title or by visible text)
        link = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "a[title='Télécharger']")
        ))

        # Capture href before click
        href_before = link.get_attribute("href")
        print("href before click:", href_before)

        # Click the link
        link.click()

        # If href is set dynamically on click, re-read it
        href_after = link.get_attribute("href")
        print("href after click:", href_after)
        sleep(10)


        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        tables = []
    
        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                #print(table)
                tables.append(table)

        for table in tables:
            for tab in table:
                
                if tab[0].startswith('ETABLISSEMENTS'):
                    continue
                # print(tab)
                name_ = tab[1].replace('\n',' ')
                address_ = tab[2]
                contacts  = tab[3]
                # print(name_)
                city_ = extract_fields(address_)['city']
                zip_ = extract_fields(address_)['postal_code']
                address_clean = extract_fields(address_)['raw_cleaned']

                phone_ = extract_contacts(contacts)['phones'][0] if len(extract_contacts(contacts)['phones'])>0 else ''
                email_ = extract_contacts(contacts)['emails'][0] if len(extract_contacts(contacts)['emails'])>0 else ''
                website_ = extract_contacts(contacts)['websites'][0] if len(extract_contacts(contacts)['websites'])>0 else ''       

                sqldict['Name'].append(name_)
                sqldict['Address_1'].append(address_clean)
                sqldict['City'].append(city_)
                sqldict['Zip'].append(zip_)
                sqldict['Phone'].append(phone_)
                sqldict['Email'].append(email_)
                sqldict['Website'].append(website_)
                sqldict['ListName'].append('Supervised entities by the Central Bank of the Republic of Guinea')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append('1')
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
    elif reg == 'GN BCRG 2':
        links = wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "a[title='Télécharger']")
        ))

        for i, link in enumerate(links, 1):
            href = link.get_attribute("href")
            print(f"{i} href:", href)
            driver.execute_script("arguments[0].click();", link)
            sleep(10)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

            if i ==1:

                with pdfplumber.open(dl_files[0]) as pdf:
                    page = pdf.pages[0]
                    tables = page.find_tables()
                    table = tables[0]

                    x0, top, x1, bottom = table.bbox
                    page_x0, page_top, page_x1, page_bottom = page.bbox

                    expanded_bbox = (x0, top, min(x1 + 200, page_x1), bottom)

                    obs_text = page.within_bbox(expanded_bbox).extract_text()

                lst = obs_text.split('\n')
                needle = "agrement retire"
                idxs = [i for i, s in enumerate(lst)
                        if needle in s.lower().replace("é", "e").replace("è", "e").replace("ê", "e")]
                row_nums = []
                for i in idxs:
                    m = re.match(r"\s*(\d+)\b", lst[i])
                    row_nums.append(m.group(1) if m else None)

                # print(idxs)      # [5, 9, 14, 26, 29]
                # print(row_nums)  # ['3', '7', '12', '24', None]

                tables_info = []

                with pdfplumber.open(dl_files[0]) as pdf:
                    for page in pdf.pages:
                        table = page.extract_table()
                        #print(table)
                        tables_info.append(table)

                for table in tables_info:
                    for index, tab in enumerate(table[:-1]):
                        # print(index+1)
                        if index+1 not in idxs:
                                if index !=0 and tab[1]:
                                    name_ = tab[1]
                                    country   = tab[4]
                                    tel_ = tab[5]
                                    email_ = tab[6]
                                    # print(name_.replace('\n',' '))
                                    # print(country)
                                    # print(tel_)
                                    sqldict['Name'].append(name_)
                                    sqldict['Phone'].append(tel_)
                                    sqldict['City'].append(country)
                                    sqldict['ListName'].append('Supervised entities by the Central Bank of the Republic of Guinea')
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['RegCtry'].append(reg.split()[0])
                                    sqldict['RegCode'].append(reg.split()[1])
                                    sqldict['ListCode'].append('1')
                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)
                    

            elif i == 2:
                tess_dir = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR"
                os.environ["PATH"] = tess_dir + ";" + os.environ.get("PATH", "")
                ocr = TesseractOCR(n_threads=1, lang="eng")
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
                doc = PDF(dl_files[0])
                extracted_tables = doc.extract_tables(ocr=ocr,
                                    implicit_rows=False,
                                    implicit_columns=False,
                                    borderless_tables=False,
                                    min_confidence=55)

                for page, tables in extracted_tables.items():
                    for idx, table in enumerate(tables):
                        for index, row in enumerate(table.content.values()):
                            #print(f"Row {index}:")
                            if index ==0:
                                continue
                            #print(row[0].value.replace("\n", " ") if row[0].value else "")
                            eme_ =row[1].value.replace("\n", " ") if row[1].value else ""    
                            nom_product = row[2].value.replace("\n", " ") if row[2].value else ""
                            date_register = row[3].value.replace("\n", " ") if row[3].value else ""   
                            if eme_:
                                name_ = eme_
                            else:
                                name_ = nom_product
                            
                            date_register = date_register.replace('\n','').strip('|')
                                    
                            sqldict['Name'].append(name_)
                            sqldict['RegulationDate'].append(date_register.split('DU')[-1].strip())
                            sqldict['ListName'].append('Supervised entities by the Central Bank of the Republic of Guinea')
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegCtry'].append(reg.split()[0])
                            sqldict['RegCode'].append(reg.split()[1])
                            sqldict['ListCode'].append('1')
                            sqldict['RegulationType'].append('Regulated')
                            sqldict = bourange_same_length_array(sqldict)

            
            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))
                
    elif reg == 'GN BCRG 3':
        # Locate the download link (by title or by visible text)
        link = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "a[title='Télécharger']")
        ))

        # Capture href before click
        href_before = link.get_attribute("href")
        print("href before click:", href_before)

        # Click the link
        link.click()

        # If href is set dynamically on click, re-read it
        href_after = link.get_attribute("href")
        print("href after click:", href_after)
        sleep(10)


        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        tables = []


        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                #print(table)
                tables.append(table)
        for table in tables:
            if not table:
                continue
            for tab in table[1:]:
                if tab and len(tab) > 2:
                    name_ = tab[1]
                    address_ = tab[2]
                    #print(name_)

                    sqldict['Name'].append(name_)
                    sqldict['ListProcessDate'].append(processdate)        
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append('1')
                    sqldict['RegulationType'].append('Regulated')
                    # Split into records by blank line
                    records = [r.strip() for r in address_.split("\n\n") if r.strip()]

                    def parse_record(r):
                        out = {"address": "", "bp": "", "email": "", "site": "", "tel": ""}
                        # normalize separators
                        r = r.replace(";", "\n").replace("Email ;", "Email :")
                        lines = [l.strip() for l in r.splitlines() if l.strip()]

                        # grab fields by keyword
                        for line in lines:
                            if re.search(r"\bEmail\b", line, re.I):
                                out["email"] = re.sub(r".*Email\s*[:\-]?\s*", "", line, flags=re.I).strip()
                            elif re.search(r"\bSite\b", line, re.I):
                                out["site"] = re.sub(r".*Site\s*[:\-]?\s*", "", line, flags=re.I).strip()
                            elif re.search(r"\bTel\b", line, re.I):
                                out["tel"] = re.sub(r".*Tel\s*[:\-]?\s*", "", line, flags=re.I).strip()
                            elif re.search(r"\bB\.?P\.?\b", line, re.I):
                                out["bp"] = re.sub(r".*B\.?P\.?\s*[:\-]?\s*", "", line, flags=re.I).strip()
                            else:
                                out["address"] += (" " if out["address"] else "") + line
                        return out

                    for r in records:
                        parsed = parse_record(r)
                        address_clean = parsed["address"]
                        email_ = parsed["email"]
                        website_ = parsed["site"]
                        phone_ = parsed["tel"]
                        zip_ = parsed["bp"]

                        sqldict['Address_1'].append((address_clean + ' ' + zip_).strip())

                        m = re.search(r"\b\d{4}\b", zip_ or "")
                        sqldict['Zip'].append(m.group(0) if m else "")

                        sqldict['Phone'].append(phone_)
                        sqldict['Email'].append(email_)
                        sqldict['Website'].append(website_)
                        sqldict['ListName'].append('Supervised entities by the Central Bank of the Republic of Guinea')
                        sqldict = bourange_same_length_array(sqldict)


    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
                        

                        
    

    

Working with GN BCRG 1
href before click: https://www.bcrg-guinee.org/wp-content/uploads/2023/03/BANQUES-AGREEES.pdf
href after click: https://www.bcrg-guinee.org/wp-content/uploads/2023/03/BANQUES-AGREEES.pdf
Working with GN BCRG 2
1 href: https://www.bcrg-guinee.org/wp-content/uploads/2020/02/Adresses-des-IMF.pdf
2 href: https://www.bcrg-guinee.org/wp-content/uploads/2022/10/Liste-des-EME.pdf
Working with GN BCRG 3
href before click: https://www.bcrg-guinee.org/wp-content/uploads/2020/02/societe-assurances.pdf
href after click: https://www.bcrg-guinee.org/wp-content/uploads/2020/02/societe-assurances.pdf


In [7]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 61 values.
Key 'priority' has 61 values.
Key 'ListLabel' has 61 values.
Key 'Typology' has 61 values.
Key 'EntryType' has 61 values.
Key 'Name' has 61 values.
Key 'InternalID_1' has 61 values.
Key 'InternalID_1_type' has 61 values.
Key 'InternalID_2' has 61 values.
Key 'InternalID_2_type' has 61 values.
Key 'InternalID_3' has 61 values.
Key 'InternalID_3_type' has 61 values.
Key 'CoType' has 61 values.
Key 'License_Type' has 61 values.
Key 'Address_1' has 61 values.
Key 'Address_2' has 61 values.
Key 'City' has 61 values.
Key 'Zip' has 61 values.
Key 'Cntry' has 61 values.
Key 'Phone' has 61 values.
Key 'Fax' has 61 values.
Key 'Website' has 61 values.
Key 'Email' has 61 values.
Key 'RegulationType' has 61 values.
Key 'RegulationTypeCode' has 61 values.
Key 'RegulationDate' has 61 values.
Key 'CancellationDate' has 61 values.
Key 'RegCtry' has 61 values.
Key 'RegCode' has 61 values.
Key 'ListCode' has 61 values.
Key 'ListLanguage' has 61 values.
Key 'ListValidityDate' h

In [8]:
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

sleep(3)

driver.quit()